In [ ]:
# Copyright 2025 Google LLC
#
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
#     https://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

# Module 03: BigQuery Data Engineering Agent for Data Warehouse code generation

This notebook contains the prep needed to build the data warehouse off of the code generated by the BigQuery Data Engineering Agent.

Programmatic invocation of the Data Engineering Agent is currently not supported, we will therefore do the pre-work here and switch to the UI to generate the Data Warehouse code

## 1. Foundations

### 1.1. Installs

In [ ]:
!pip install google-cloud-dataplex==2.11.0  -q
!pip install google-cloud-dataform==0.6.2 -q

### 1.2. Variable initialization & imports

In [ ]:
PROJECT_ID_LIST=!gcloud config list --format "value(core.project)" 2>/dev/null
PROJECT_ID=PROJECT_ID_LIST[0]
LOCATION="us-central1"
OLTP_DATASET_ID="rscw_oltp_stg_ds"
DWH_DATASET_ID="rscw_dwh_ds"
OLTP_METADATA_DATASET_ID="rscw_oltp_metadata_ds"
DWH_METADATA_DATASET_ID="rscw_dwh_metadata_ds"
DATAFORM_REPO="rscw-df-repo"
DATAFORM_WORKSPACE="rscw-df-ws"
OLTP_DATASET_RESOURCE_URI = f"//bigquery.googleapis.com/projects/{PROJECT_ID}/datasets/{OLTP_DATASET_ID}"
DWH_DATASET_RESOURCE_URI = f"//bigquery.googleapis.com/projects/{PROJECT_ID}/datasets/{DWH_DATASET_ID}"
BASE_URL_FOR_DATAPLEX_SCAN="https://dataplex.googleapis.com/v1"
BASE_URL_FOR_DATA_ENGINEERING_AGENT = "https://geminidataanalytics.googleapis.com"



In [ ]:
import re
import time
import json
import os
import io
import base64
import yaml
import pandas as pd
import pandas_gbq
import requests
import datetime
import decimal
import google.auth
import google.auth.transport.requests
import pyarrow as pa
import pyarrow.parquet as pq
import psycopg2
import google.genai
import git
import traceback
import subprocess
import shutil
import configparser

from typing import List, Dict, Any, Optional
from google.cloud import bigquery
from google.cloud import dataplex_v1
from google.cloud import dataform_v1beta1
from google.api_core import exceptions, operation
from google.cloud.exceptions import NotFound
from google.cloud.exceptions import Conflict
from google.protobuf.timestamp_pb2 import Timestamp
from google.genai.types import CreateBatchJobConfig, JobState
from google.cloud import storage
from google.cloud import secretmanager
from google.adk.agents import Agent
from google.adk.tools import ToolContext

/usr/local/lib/python3.12/dist-packages/google/cloud/aiplatform/models.py:52: FutureWarning: Support for google-cloud-storage < 3.0.0 will be removed in a future version of google-cloud-aiplatform. Please upgrade to google-cloud-storage >= 3.0.0.
  from google.cloud.aiplatform.utils import gcs_utils


## 2. Util functions & setup

### 2.1. General functions

In [ ]:
def get_auth_token():
    creds, _ = google.auth.default()
    # Refresh the credentials to get an access token
    creds.refresh(google.auth.transport.requests.Request())
    return creds.token

def sanitize_string_with_hyphens(input_string):
    """
    Converts a string to lowercase and replaces any character that is not
    a lowercase letter or a number with a hyphen.
    """
    # convert the entire string to lowercase.
    processed_string = input_string.lower()

    # The pattern [^a-z0-9] matches any single character that is NOT
    # a lowercase letter (a-z) or a digit (0-9).
    sanitized_string = re.sub(r'[^a-z0-9]', '-', processed_string)

    return sanitized_string


### 2.2. BQ utils

In [ ]:
def truncate_bigquery_table(bq_table_uri: str):
    """
    Deletes all rows from a specified BigQuery table.

    This function executes a TRUNCATE TABLE DML statement, which is an
    efficient way to clear a table.

    Returns:
        A string confirming the successful truncation or describing an error.
    """
    try:
        # Construct a BigQuery client object.
        client = bigquery.Client()

        # Sanitize the table URI by wrapping it in backticks.
        safe_table_uri = f"`{bq_table_uri}`"

        # Construct the DML query to truncate the table.
        truncate_query = f"TRUNCATE TABLE {safe_table_uri}"

        # Execute the query.
        print(f"Executing query: {truncate_query}")
        query_job = client.query(truncate_query)

        # Wait for the DML query to complete.
        query_job.result()

        return f"Successfully truncated table {bq_table_uri}"

    except exceptions.NotFound:
        return f"Error: The table '{bq_table_uri}' was not found."
    except Exception as e:
        return f"An unexpected error occurred: {e}"

def read_bigquery_table(
    project_id: str,
    dataset_id: str,
    table_id: str
) -> pd.DataFrame:
    """
    Reads all rows from a BigQuery table and returns them as a pandas DataFrame.

    This function authenticates using the environment's default credentials and uses
    the BigQuery Storage Read API for efficient data retrieval.

    Returns:
        A pandas DataFrame containing all rows from the table.
        Returns an empty DataFrame if the table is not found or an error occurs.
    """
    # Construct a BigQuery client object.
    # The client library will automatically handle authentication.
    client = bigquery.Client(project=project_id)

    # Construct the full table ID in the format `project.dataset.table`.
    table_ref = f"{project_id}.{dataset_id}.{table_id}"

    try:
        # Use the list_rows() method to get a row iterator from the API.
        # The to_dataframe() method downloads all rows and converts them to a DataFrame.
        print(f"Reading all rows from table: {table_ref}...")
        rows = client.list_rows(table_ref)
        dataframe = rows.to_dataframe()
        print(f"Successfully read {len(dataframe)} rows.")
        return dataframe

    except Exception as e:
        print(f"An error occurred: {e}")
        # Return an empty DataFrame in case of an error.
        return pd.DataFrame()

def write_dict_to_bigquery(bq_table_uri: str, data_to_insert: dict, delete_conditions: dict = None):
    """
    Writes a row of data to a BigQuery table.

    Optionally deletes rows from the table based on a condition before inserting new data.
    Create the dataset if it doesn't exist.
    Create the table if it doesn't exist, using the data's keys as the schema.
    Truncate the table if it exists.
    Update the table schema if the data contains new columns.
    Insert the data as a new row.

    Returns:
        A string indicating success or failure.
    """
    if not data_to_insert:
        return "No data provided to write to BigQuery."

    try:
        # Construct a BigQuery client object.
        client = bigquery.Client()

        # Parse the table URI.
        project_id, dataset_id, table_id = bq_table_uri.split('.')
        dataset_ref = client.dataset(dataset_id)
        table_ref = dataset_ref.table(table_id)
        table_ref_str = f"{project_id}.{dataset_id}.{table_id}"

        # Create dataset if it doesn't exist.
        try:
            client.get_dataset(dataset_ref)
        except exceptions.NotFound:
            print(f"Dataset {dataset_id} not found. Creating it in us-central1.")
            dataset = bigquery.Dataset(dataset_ref)
            dataset.location = "us-central1"
            client.create_dataset(dataset, exists_ok=True)

        # Prepare the schema based on the data to insert.
        new_schema_fields = [
            bigquery.SchemaField(key, "STRING") for key in data_to_insert
        ]

        # Check if table exists and update schema if necessary.
        try:
            table = client.get_table(table_ref)
            current_schema = {field.name for field in table.schema}
            new_schema = list(table.schema)

            for field in new_schema_fields:
                if field.name not in current_schema:
                    new_schema.append(field)

            table.schema = new_schema
            client.update_table(table, ["schema"])

        except exceptions.NotFound:
            print(f"Table {table_id} not found. Creating it.")
            table = bigquery.Table(table_ref_str, schema=new_schema_fields)
            client.create_table(table)

        # Delete rows if conditions are provided.
        if delete_conditions and table_id in ["tables", "columns"]:
            where_clauses = []
            for condition in delete_conditions:
                column = condition.get('column')
                value = condition.get('value')
                if column and value is not None:
                    where_clauses.append(f"{column} = '{value}'")

            if where_clauses:
                delete_query = f"DELETE FROM {table_ref_str} WHERE " + " AND ".join(where_clauses)
                query_job = client.query(delete_query)
                query_job.result()  # Wait for the job to complete.

        # Insert the new row.
        row_to_insert = {key: str(value) for key, value in data_to_insert.items()}
        errors = client.insert_rows_json(table_ref_str, [row_to_insert])

        if not errors:
            return f"Successfully wrote data to {table_ref_str}"
        else:
            return f"Failed to insert rows: {errors}"

    except Exception as e:
        return f"An unexpected error occurred: {e}"

def update_bigquery_metadata(
    project_id: str,
    dataset_id: str,
    new_description: str,
    table_id: str = None,
    column_name: str = None
):
  """
  Updates the description of a BigQuery dataset, table, or column.
  """
  try:
    client = bigquery.Client(project=project_id)

    if table_id and column_name:
      # Update a column's description
      dataset_ref = client.dataset(dataset_id)
      table_ref = dataset_ref.table(table_id)
      table = client.get_table(table_ref)

      new_schema = []
      column_found = False
      for field in table.schema:
        if field.name == column_name:
          column_found = True
          # Recreate the SchemaField with the new description
          new_field = field.to_api_repr()
          new_field['description'] = new_description
          new_schema.append(bigquery.SchemaField.from_api_repr(new_field))
        else:
          new_schema.append(field)

      if not column_found:
        print(f"Error: Column '{column_name}' not found in table '{table_id}'.")
        return

      table.schema = new_schema
      client.update_table(table, ["schema"])
      print(f"Successfully updated description for column: {column_name} in table {project_id}.{dataset_id}.{table_id}")

    elif table_id:
      # Update a table's description
      dataset_ref = client.dataset(dataset_id)
      table_ref = dataset_ref.table(table_id)
      table = client.get_table(table_ref)
      table.description = new_description
      client.update_table(table, ["description"])
      print(f"Successfully updated description for table: {project_id}.{dataset_id}.{table_id}")

    else:
      # Update a dataset's description
      dataset_ref = client.dataset(dataset_id)
      dataset = client.get_dataset(dataset_ref)
      dataset.description = new_description
      client.update_dataset(dataset, ["description"])
      print(f"Successfully updated description for dataset: {project_id}.{dataset_id}")

  except NotFound as e:
    print(f"Error: Resource not found. Please check your IDs. Details: {e}")
  except Exception as e:
    print(f"An error occurred: {e}")

def get_dataset_tables(dataset_id):
  """
  Fetches a list of tables within a specified BigQuery dataset.
  This function initializes a BigQuery client for a predefined project
  and retrieves an iterator for the tables in the given dataset.
  """
  client = bigquery.Client(project=f"{PROJECT_ID}")
  tables = client.list_tables(dataset_id)
  return tables

### 2.3. Dataform utils

In [ ]:

def delete_dataform_workspace(token: str, project_id: str, location: str, repository_id: str, workspace_id: str):
    """
    Deletes a Dataform workspace.
    """

    headers = {
        "Authorization": f"Bearer {token}",
    }
    workspace_url = f"https://dataform.googleapis.com/v1beta1/projects/{project_id}/locations/{location}/repositories/{repository_id}/workspaces/{workspace_id}"

    print(f"Attempting to delete workspace '{workspace_id}'...")
    try:
        response = requests.delete(workspace_url, headers=headers)
        response.raise_for_status()
        print(f"Workspace '{workspace_id}' deleted successfully.")
    except requests.exceptions.HTTPError as e:
        if e.response.status_code == 404:
            print(f"Workspace '{workspace_id}' not found.")
        else:
            print(f"Failed to delete workspace: {e.response.text}")

def create_dataform_repository_and_workspace(token, project_id, location, repository_id, workspace_id):
    """
    Checks for a Dataform repository, creates it if it doesn't exist,
    and then creates a workspace in it.
    """

    headers = {
        "Authorization": f"Bearer {token}",
        "Content-Type": "application/json",
    }

    repo_url = f"https://dataform.googleapis.com/v1beta1/projects/{project_id}/locations/{location}/repositories/{repository_id}"

    # Check if the repository exists
    repo_check_response = requests.get(repo_url, headers=headers)

    repo_exists = False
    if repo_check_response.status_code == 200:
        print(f"Repository '{repository_id}' already exists.")
        repo_exists = True
    elif repo_check_response.status_code == 404:
        print(f"Repository '{repository_id}' not found. Creating it...")
        # Create the repository since it doesn't exist
        create_repo_url = f"https://dataform.googleapis.com/v1beta1/projects/{project_id}/locations/{location}/repositories?repositoryId={repository_id}"
        repo_create_response = requests.post(create_repo_url, headers=headers)

        if repo_create_response.status_code == 200:
            print(f"Repository '{repository_id}' created successfully.")
            repo_exists = True
        else:
            print(f"Failed to create repository: {repo_create_response.text}")
    else:
        # Handle other potential errors during the check
        print(f"Error checking repository: {repo_check_response.text}")

    # If the repository exists (either pre-existing or newly created), create the workspace
    if repo_exists:
        # Corrected the workspace URL construction
        workspace_url = f"{repo_url}/workspaces?workspaceId={workspace_id}"
        workspace_response = requests.post(workspace_url, headers=headers)

        if workspace_response.status_code == 200:
            print(f"Workspace '{workspace_id}' created successfully.")
        elif workspace_response.status_code == 409:
             print(f"Workspace '{workspace_id}' already exists.")
        else:
            print(f"Failed to create workspace: {workspace_response.text}")



def initialize_dataform_workspace(token, project_id, location, repository_id, workspace_id):
    """
    Initializes a Dataform workspace by creating initial files and installing packages.
    """

    headers = {
        "Authorization": f"Bearer {token}",
        "Content-Type": "application/json",
    }

    workspace_path = f"projects/{project_id}/locations/{location}/repositories/{repository_id}/workspaces/{workspace_id}"
    base_url = f"https://dataform.googleapis.com/v1beta1/{workspace_path}"

    # 1. Create workflow_settings.yaml
    workflow_settings = {
        'defaultProject': project_id,
        'defaultLocation': location,
        'defaultDataset': 'dataform',
        'defaultAssertionDataset': 'dataform_assertions',
        'dataformCoreVersion': '3.0.16',
    }
    yaml_string = yaml.dump(workflow_settings)
    # The API expects the file content to be a base64 encoded string
    encoded_yaml = base64.b64encode(yaml_string.encode('utf-8')).decode('utf-8')

    write_file_url = f"{base_url}:writeFile"
    write_settings_payload = {
        "path": "workflow_settings.yaml",
        "contents": encoded_yaml
    }

    settings_response = requests.post(write_file_url, headers=headers, json=write_settings_payload)

    if settings_response.status_code == 200:
        print("Successfully wrote workflow_settings.yaml.")
    else:
        print(f"Failed to write workflow_settings.yaml: {settings_response.text}")
        return

    # 2. Create .gitignore
    gitignore_content = "node_modules/"
    encoded_gitignore = base64.b64encode(gitignore_content.encode('utf-8')).decode('utf-8')
    write_gitignore_payload = {
        "path": ".gitignore",
        "contents": encoded_gitignore
    }
    gitignore_response = requests.post(write_file_url, headers=headers, json=write_gitignore_payload)

    if gitignore_response.status_code == 200:
        print("Successfully wrote .gitignore.")
    else:
        print(f"Failed to write .gitignore: {gitignore_response.text}")
        return

    # 3. Install npm packages
    print("Installing npm packages...")
    install_packages_url = f"{base_url}:installNpmPackages"
    packages_response = requests.post(install_packages_url, headers=headers)

    if packages_response.status_code == 200:
        print("NPM packages installed successfully.")
    else:
        print(f"Failed to install NPM packages: {packages_response.text}")

def get_dataform_workspace_contents(
    token: str,
    project_id: str,
    location: str,
    repository_id: str,
    workspace_id: str,
    dataform_dir: str = "definitions",
):
    """
    Lists files in a Dataform workspace's 'definitions' folder, reads their content,
    and provides an explanation.

    This function recursively lists all files within the 'definitions' directory,
    reads the content of each one, decodes it, and generates a high-level summary
    of what the SQLX file does based on its configuration and query.

    Returns:
        A list of dictionaries, where each dictionary contains the file path, its
        decoded content, and an explanation. Returns None on failure.
    """

    headers = {
        "Authorization": f"Bearer {token}",
        "Content-Type": "application/json",
    }

    try:
        print(f"Querying Dataform workspace files in '{dataform_dir}/' folder...")
        all_files = list_all_dataform_files(token, dataform_dir)
    except requests.exceptions.HTTPError as e:
        print(f"API Error while listing files: {e}\nResponse body: {e.response.text}")
        return None

    # Read the content of each file
    file_contents = []
    read_endpoint_base = (
        f"https://dataform.googleapis.com/v1/projects/{project_id}/"
        f"locations/{location}/repositories/{repository_id}/workspaces/{workspace_id}:readFile"
    )

    for file_path in all_files:
        try:
            response = requests.get(read_endpoint_base, headers=headers, params={"path": file_path})
            response.raise_for_status()

            encoded_content = response.json().get("fileContents", "")
            decoded_content = base64.b64decode(encoded_content).decode("utf-8")

            explanation = "Could not determine the purpose of this file."
            if file_path.endswith(".sqlx"):
                config_type_match = re.search(r'type:\s*["\'](\w+)["\']', decoded_content)
                action_type = config_type_match.group(1) if config_type_match else "action"
                refs = re.findall(r"ref\(['\"]([\w_]+)['\"]\)", decoded_content)

                if refs:
                    explanation = f"This file defines a new {action_type} that depends on the following source(s): {', '.join(refs)}."
                else:
                    explanation = f"This file defines a new {action_type}."

            file_contents.append({
                "file_path": file_path,
                "content": decoded_content,
                "explanation": explanation
            })

        except requests.exceptions.HTTPError as e:
            print(f"API Error reading file '{file_path}': {e}")
            continue
        except Exception as e:
            print(f"An error occurred processing file '{file_path}': {e}")
            continue

    return file_contents

def execute_dataform_pipeline(
    token: str,
    project_id: str,
    location: str,
    repository_id: str,
    workspace_id: str,
    service_account_email: str,
    target_table_name: str = ""
)-> dict:
    """
    Compiles a Dataform workspace, executes the pipeline, and waits for it to complete.

    If a target_table_name is provided, only actions tagged with that name will be executed.

    Returns:
        The final workflow invocation object, or None if any step fails.
    """

    headers = {
        "Authorization": f"Bearer {token}",
        "Content-Type": "application/json",
    }

    # --- Step 1: Compile the workspace ---
    print("Compiling Dataform workspace...")
    compilation_endpoint = (
        f"https://dataform.googleapis.com/v1/projects/{project_id}/"
        f"locations/{location}/repositories/{repository_id}/compilationResults"
    )
    workspace_resource_name = (
        f"projects/{project_id}/locations/{location}/"
        f"repositories/{repository_id}/workspaces/{workspace_id}"
    )
    compilation_body = {"workspace": workspace_resource_name}

    try:
        response = requests.post(compilation_endpoint, headers=headers, json=compilation_body)
        response.raise_for_status()
        compilation_result = response.json()
        compilation_result_name = compilation_result.get("name")
        print(f"Successfully compiled. Result name: {compilation_result_name}")
    except requests.exceptions.HTTPError as e:
        print(f"API Error during compilation: {e}\nResponse body: {e.response.text}")
        return None

    # --- Step 2: Execute the compiled result ---
    print("Executing the pipeline...")
    invocation_endpoint = (
        f"https://dataform.googleapis.com/v1/projects/{project_id}/"
        f"locations/{location}/repositories/{repository_id}/workflowInvocations"
    )

    invocation_config = {
        "serviceAccount": service_account_email
    }

    # If a target_table_name is provided, include it in the invocationConfig to filter execution by tags.
    if len(target_table_name) > 0:
        invocation_config["includedTags"] = [target_table_name]
        print(f"Executing only actions tagged with: {target_table_name}")

    invocation_body = {
        "compilationResult": compilation_result_name,
        "invocationConfig": invocation_config
    }

    try:
        response = requests.post(invocation_endpoint, headers=headers, json=invocation_body)
        response.raise_for_status()
        workflow_invocation = response.json()
        invocation_name = workflow_invocation.get('name')
        print(f"Successfully started workflow invocation: {invocation_name}")
    except requests.exceptions.HTTPError as e:
        print(f"API Error during execution: {e}\nResponse body: {e.response.text}")
        return None

    # --- Step 3: Wait for the execution to complete ---
    print("\nWaiting for execution to complete...")
    status_endpoint = f"https://dataform.googleapis.com/v1/{invocation_name}"

    while True:
        try:
            status_response = requests.get(status_endpoint, headers=headers)
            status_response.raise_for_status()
            invocation_details = status_response.json()
            current_state = invocation_details.get("state")

            print(f"Current state: {current_state}")

            if current_state in ["SUCCEEDED", "FAILED", "CANCELLED"]:
                print(f"Execution finished with state: {current_state}")
                if current_state == "SUCCEEDED":
                     print("Pipeline executed successfully.")
                else:
                     print("Pipeline execution did not succeed.")
                return invocation_details

            # Wait for 2 seconds before checking the status again
            time.sleep(2)

        except requests.exceptions.HTTPError as e:
            print(f"API Error while checking status: {e}\nResponse body: {e.response.text}")
            return None


def get_executed_files_from_invocation(token: str, invocation_details: dict) -> list[str]:
    """
    Extracts the list of executed file paths from a Dataform workflow invocation object.
    This version includes detailed print statements for debugging.
    """

    headers = {
        "Authorization": f"Bearer {token}",
        "Content-Type": "application/json",
    }

    compilation_result_name = invocation_details.get("compilationResult")
    if not compilation_result_name:
        print("No compilationResult found in invocation details.")
        return []

    compilation_result_url = f"https://dataform.googleapis.com/v1/{compilation_result_name}"

    try:
        print(f"Fetching compilation result from: {compilation_result_url}")
        response = requests.get(compilation_result_url, headers=headers)
        response.raise_for_status()
        compilation_result = response.json()

        # --- New: Print the full compilation result for inspection ---
        print("\n--- Full Compilation Result ---")
        print(json.dumps(compilation_result, indent=2))
        print("-------------------------------\n")

    except requests.exceptions.HTTPError as e:
        print(f"API Error during compilation result fetch: {e}\nResponse body: {e.response.text}")
        return []

    executed_files = []
    included_tags = invocation_details.get("invocationConfig", {}).get("includedTags", [])
    print(f"Execution was filtered by the following tags: {included_tags}")

    if "compilationResultActions" in compilation_result:
        print("\n--- Analyzing Compiled Actions ---")
        for action in compilation_result["compilationResultActions"]:
            file_path = action.get("filePath", "N/A")
            action_tags = action.get("tags", [])

            print(f"Found action for file: {file_path} with tags: {action_tags}")

            if not included_tags or any(tag in action_tags for tag in included_tags):
                if "filePath" in action:
                    print(f"  -> Match found! Adding '{file_path}' to the list of executed files.")
                    executed_files.append(action["filePath"])
        print("--------------------------------\n")
    else:
        print("Warning: 'compilationResultActions' not found in the compilation result.")


    return executed_files

def execute_pipeline_code(target_tag_value: str) -> dict:
    """
    Executes the full Dataform pipeline to create whatever is defined in the
    Workspace files.
    """

    # Validate the 'dataset' parameter to ensure it's a valid choice.
    valid_target_tag_values = ["reports", ""]
    if target_tag_value not in valid_target_tag_values:
        raise ValueError(f"Invalid dataset '{target_tag_value}' provided. Please use one of {valid_target_tag_values}.")


    print("--- Tool: Executing the Dataform pipeline... ---")

    token = get_auth_token()
    details = {}
    # Execute the pipeline from the workspace
    final_invocation_details = execute_dataform_pipeline(
        token,
        PROJECT_ID,
        LOCATION,
        DATAFORM_REPO,
        DATAFORM_WORKSPACE,
        SERVICE_ACCOUNT_FOR_DATAFORM,
        target_tag_value
    )

    if final_invocation_details:
        print("\n--- Fetching executed files ---")

        dataform_directory = "definitions"

        if target_tag_value in ["reports"]:
            dataform_directory = "definitions/reports"

        executed_files = list_all_dataform_files(token, dataform_directory)

        details["invocation_details"]=final_invocation_details

        if executed_files:
            details["executed_files"]=executed_files
            print("The following files were executed:")
            for file_path in executed_files:
                print(f"- {file_path}")
        else:
            print("Could not retrieve the list of executed files.")

        print("\n--- Final Invocation Details ---")
        print(json.dumps(final_invocation_details, indent=2))
        print("------------------------------")

    return {"status": "success", "Response": final_invocation_details, "Details":executed_files}

def explain_pipeline_code() -> Dict[str, Any]:
    """
    Gets the content and explanations of all the files in the Dataform repo
    definitions folder and return them as a dictionary.
    """
    print("--- Tool: Getting explanations for the files in the workspace... ---")

    token = get_auth_token()

    workspace_contents = get_dataform_workspace_contents(
        token,
        PROJECT_ID,
        LOCATION,
        DATAFORM_REPO,
        DATAFORM_WORKSPACE
    )

    if workspace_contents:
        return {"status": "success", "Details": workspace_contents}
    else:
        print("Could not retrieve the workspace contents.")
        # Return a dictionary indicating failure and an empty list
        return {"status": "error", "An error occured. Workspace contents are empty": []}

def list_all_dataform_files(token: str, dataform_dir: str = "definitions"):
    headers = {
        "Authorization": f"Bearer {token}",
        "Content-Type": "application/json",
    }

    query_endpoint = (
        f"https://dataform.googleapis.com/v1/projects/{PROJECT_ID}/"
        f"locations/{LOCATION}/repositories/{DATAFORM_REPO}/workspaces/{DATAFORM_WORKSPACE}:queryDirectoryContents"
    )

    all_files = set()

    dirs_to_query = [dataform_dir]
    known_dirs = {dataform_dir}

    while dirs_to_query:
        path_to_query = dirs_to_query.pop(0)

        params = {"path": path_to_query}
        response = requests.get(query_endpoint, headers=headers, params=params)

        if response.status_code == 404:
            print(f"Warning: Directory not found at path: '{path_to_query}'. Skipping.")
            continue
        response.raise_for_status()

        for entry in response.json().get("directoryEntries", []):
            if "directory" in entry:
                full_path = entry['directory']
                if full_path not in known_dirs:
                    dirs_to_query.append(full_path)
                    known_dirs.add(full_path)
            elif "file" in entry:
                full_path = entry['file']
                all_files.add(full_path)

    return sorted(list(all_files))

def get_dataform_workspace_contents(
    token: str,
    project_id: str,
    location: str,
    repository_id: str,
    workspace_id: str,
    dataform_dir: str = "definitions",
):
    """
    Lists files in a Dataform workspace's 'definitions' folder, reads their content,
    and provides an explanation.

    This function recursively lists all files within the 'definitions' directory,
    reads the content of each one, decodes it, and generates a high-level summary
    of what the SQLX file does based on its configuration and query.

    Returns:
        A list of dictionaries, where each dictionary contains the file path, its
        decoded content, and an explanation. Returns None on failure.
    """

    headers = {
        "Authorization": f"Bearer {token}",
        "Content-Type": "application/json",
    }

    try:
        print(f"Querying Dataform workspace files in '{dataform_dir}/' folder...")
        all_files = list_all_dataform_files(token, dataform_dir)
    except requests.exceptions.HTTPError as e:
        print(f"API Error while listing files: {e}\nResponse body: {e.response.text}")
        return None

    # Read the content of each file
    file_contents = []
    read_endpoint_base = (
        f"https://dataform.googleapis.com/v1/projects/{project_id}/"
        f"locations/{location}/repositories/{repository_id}/workspaces/{workspace_id}:readFile"
    )

    for file_path in all_files:
        try:
            response = requests.get(read_endpoint_base, headers=headers, params={"path": file_path})
            response.raise_for_status()

            encoded_content = response.json().get("fileContents", "")
            decoded_content = base64.b64decode(encoded_content).decode("utf-8")

            explanation = "Could not determine the purpose of this file."
            if file_path.endswith(".sqlx"):
                config_type_match = re.search(r'type:\s*["\'](\w+)["\']', decoded_content)
                action_type = config_type_match.group(1) if config_type_match else "action"
                refs = re.findall(r"ref\(['\"]([\w_]+)['\"]\)", decoded_content)

                if refs:
                    explanation = f"This file defines a new {action_type} that depends on the following source(s): {', '.join(refs)}."
                else:
                    explanation = f"This file defines a new {action_type}."

            file_contents.append({
                "file_path": file_path,
                "content": decoded_content,
                "explanation": explanation
            })

        except requests.exceptions.HTTPError as e:
            print(f"API Error reading file '{file_path}': {e}")
            continue
        except Exception as e:
            print(f"An error occurred processing file '{file_path}': {e}")
            continue

    return file_contents

def delete_dataform_folder(
    token: str,
    project_id: str,
    location: str,
    repository_id: str,
    workspace_id: str,
    folder: str
):
    """
    Deletes a Dataform folder only if it exists.
    """

    if folder is None:
        raise ValueError("The 'folder' parameter cannot be None.")

    headers = {
        "Authorization": f"Bearer {token}",
        "Content-Type": "application/json",
    }
    instruction_path = ".gdeagent/instructions"

    # --- Step 1: Check if the directory exists before trying to delete it ---
    #queryDirectoryContents will return a 404 if the path does not exist.
    check_endpoint = (
        f"https://dataform.googleapis.com/v1beta1/projects/{project_id}/"
        f"locations/{location}/repositories/{repository_id}/workspaces/{workspace_id}:queryDirectoryContents"
    )

    print(f"Checking for existence of directory: '{folder}'...")
    try:
        check_response = requests.get(check_endpoint, headers=headers, params={"path": folder})

        # A 404 status code means the directory was not found.
        if check_response.status_code == 404:
            print(f"Directory '{folder}' not found. Nothing to delete.")
            return  # Exit the function gracefully

        # Raise an exception for any other HTTP errors during the check.
        check_response.raise_for_status()

    except requests.exceptions.HTTPError as e:
        print(f"An error occurred while checking for the directory: {e}")
        return

    # --- Step 2: If the check was successful, the directory exists, so we delete it ---
    print(f"Directory '{folder}' found. Proceeding with deletion...")
    delete_endpoint = (
        f"https://dataform.googleapis.com/v1beta1/projects/{project_id}/"
        f"locations/{location}/repositories/{repository_id}/workspaces/{workspace_id}:removeDirectory"
    )
    body = {"path": folder}

    try:
        response = requests.post(delete_endpoint, headers=headers, json=body)
        response.raise_for_status()  # Raise an exception for bad status codes during deletion
        print(f"Successfully deleted directory: '{folder}'")
        return response.json()
    except requests.exceptions.HTTPError as e:
        print(f"An error occurred during deletion: {e}")
        # Re-raise the exception to be handled by the caller if deletion fails
        raise e

def create_dataform_instructions_file(
    token: str,
    project_id: str,
    location: str,
    repository_id: str,
    workspace_id: str,
    filename: str,
    content: str
):
    """
    Creates a Dataform instruction file with the given content.

    Returns:
        The JSON response from the Dataform API.
    """

    # The Dataform API endpoint for writing a file
    # This is based on the v1beta1 API version
    api_endpoint = (
        f"https://dataform.googleapis.com/v1beta1/projects/{project_id}/"
        f"locations/{location}/repositories/{repository_id}/workspaces/{workspace_id}:writeFile"
    )

    # Instruction files must be placed in the .gdeagent/instructions/ directory
    # No subdirectories are allowed.
    instruction_path = f".gdeagent/instructions/{filename}"

    # The content of the file must be base64 encoded
    encoded_content = base64.b64encode(content.encode("utf-8")).decode("utf-8")

    headers = {
        "Authorization": f"Bearer {token}",
        "Content-Type": "application/json",
    }

    body = {
        "path": instruction_path,
        "contents": encoded_content,
    }

    response = requests.post(api_endpoint, headers=headers, json=body)
    response.raise_for_status()  # Raise an exception for bad status codes

    return response.json()

### 2.4. Dataplex utils


In [ ]:
def describe_dataset(description_df: pd.DataFrame) -> str:
    """
    Converts the dataset description DataFrame into a human-readable string.

    Returns:
        A string containing the dataset's description.
    """
    try:
        # Extracts the first description from the DataFrame
        description = description_df["dataset_description"].iloc[0]
        return f"Dataset Overview:\n{description}"
    except (IndexError, KeyError):
        return "No dataset description was found."

def describe_relationships(relationships_df: pd.DataFrame) -> List[str]:
    """
    Converts the relationships DataFrame into human-readable sentences.

    Returns:
        A list of strings, where each string describes a table relationship.
    """
    descriptions = []
    for _, row in relationships_df.iterrows():
        # Constructs a sentence describing the join between two tables
        sentence = (
            f"Table '{row['table_1']}' connects to table '{row['table_2']}' by joining "
            f"'{row['table_1']}.{row['table_1_column']}' with '{row['table_2']}.{row['table_2_column']}'."
        )
        descriptions.append(sentence)
    return descriptions

def describe_tables(tables_df: pd.DataFrame) -> List[str]:
    """
    Converts the tables DataFrame into human-readable descriptions.

    Returns:
        A list of strings, where each string is a description of a table.
    """
    descriptions = []
    for _, row in tables_df.iterrows():
        # Formats a description for each table
        description = f"Table '{row['name']}': {row['description']}"
        descriptions.append(description)
    return descriptions

def describe_columns(columns_df: pd.DataFrame) -> str:
    """
    Converts the columns DataFrame into a structured, human-readable text block.

    Returns:
        A single string that describes the columns for each table.
    """
    full_description = "Column Details for Each Table:\n"
    # Group the DataFrame by table name to process each table's columns together
    for table_name, group in columns_df.groupby("table_name"):
        full_description += f"\n--- Table: {table_name} ---\n"
        for _, row in group.iterrows():
            # Add a formatted line for each column's name and description
            full_description += f"- {row['column_name']}: {row['column_description']}\n"
    return full_description


def get_scan_results(token,scan_id):
    """
    Retrieves the results of the data scan.
    """
    print("Fetching scan results...")

    url = f"{BASE_URL_FOR_DATAPLEX_SCAN}/projects/{PROJECT_ID}/locations/{LOCATION}/dataScans/{scan_id}?view=FULL"
    print(f"Base URL for API call for full results:{url}")
    headers = {"Authorization": f"Bearer {token}"}
    response = requests.get(url, headers=headers)
    response.raise_for_status()
    print("Successfully fetched scan results.")
    return response.json()


def persist_dataplex_scan_output_to_bq_tables(dataset_type: str) -> dict:
    """
    Saves to BigQuery the Data Insights scans (knowledge and documentation)
    for a specified dataset. The dataset can be the operational dataset or the star schema
    dataset, also know as data warehouse dataset.

    Valid inputs for 'dataset_type' are 'OLTP' or 'OLAP'.
    """

    # Validate the 'dataset' parameter to ensure it's a valid choice.
    valid_dataset_types = ["OLTP", "OLAP"]
    if dataset_type not in valid_dataset_types:
        raise ValueError(f"Invalid dataset '{dataset_type}' provided. Please use one of {valid_dataset_types}.")

    print(f"Starting saving the Data Insights scans for the {dataset_type} dataset... ---")

    token = get_auth_token()

    # 2. Set the correct constants based on the dataset parameter.

    table_scan_suffix = "table-documentation-scan"

    if dataset_type == "OLTP":
        dataset_resource = OLTP_DATASET_RESOURCE_URI
        knowledge_scan_id = "rscw-oltp-stg-ds-dataset-documentation-scan"
        dataset_id = OLTP_DATASET_ID
        metadata_dataset_id = OLTP_METADATA_DATASET_ID
    else:  # This will be 'datawarehouse' due to the validation above
        dataset_resource = DWH_DATASET_RESOURCE_URI
        knowledge_scan_id = "rscw-dwh-ds-dataset-documentation-scan"
        dataset_id = DWH_DATASET_ID
        metadata_dataset_id = DWH_METADATA_DATASET_ID

    try:

        relationships = {}
        details = {}

        # Fetch the knowledge scan
        print(f"Fetching dataset documentation scan ---")
        results = get_scan_results(token, knowledge_scan_id)
        print(f"Results:{results}")


        # Part 1: Dataset description
        print("Capturing and persisting Dataset description")
        datasetDescription = results.get("dataDocumentationResult", {}).get("datasetResult", {}).get("overview")
        details["Description"] = datasetDescription

        data = {
                "dataset_description": datasetDescription
        }

        print(f"datasetDescription={datasetDescription}")
        if datasetDescription is not None:
            truncate_bigquery_table(f"{PROJECT_ID}.{metadata_dataset_id}.dataset_description")
            write_dict_to_bigquery(f"{PROJECT_ID}.{metadata_dataset_id}.dataset_description", data)
            update_bigquery_metadata(PROJECT_ID, dataset_id, datasetDescription)
            print("PART 1 - Persisted dataset description")


        # Part 2: Table relationships
        print("Capturing and persisting Table relationships")
        schemaRelationships = results.get("dataDocumentationResult", {}).get("datasetResult", {}).get("schemaRelationships")
        details["Relationships"] = schemaRelationships
        if schemaRelationships is not None:
            truncate_bigquery_table(f"{PROJECT_ID}.{metadata_dataset_id}.dataset_table_relationships")

        # Use a 'for' loop to iterate over each element in the list
        for i, relationship in enumerate(schemaRelationships):
            # Now 'relationship' is one of the dictionaries from the list

            # Safely access the data inside each dictionary
            join_type = relationship.get("type", "Unknown Type").replace("SCHEMA_JOIN", "JOIN")

            # Reset
            left_table = "N/A"
            left_column = "N/A"
            right_table = "N/A"
            right_column = "N/A"

            # Access elements of interest
            left_table_fqn = relationship.get("leftSchemaPaths", []).get("tableFqn", {})
            left_table_resource_uri_parts = left_table_fqn.split("/")
            left_table=left_table_resource_uri_parts[8]
            left_table_column = relationship.get("leftSchemaPaths", []).get("paths", {})[0]

            right_table_fqn = relationship.get("rightSchemaPaths", []).get("tableFqn", {})
            right_table_resource_uri_parts = right_table_fqn.split("/")
            right_table=  right_table_resource_uri_parts[8]
            right_table_column = relationship.get("rightSchemaPaths", []).get("paths", {})[0]

            row_data = {
                "table_1": left_table,
                "table_1_column": left_table_column,
                "table_2": right_table,
                "table_2_column": right_table_column,
                "join_type": join_type
            }

            write_dict_to_bigquery(f"{PROJECT_ID}.{metadata_dataset_id}.dataset_table_relationships", row_data)


        #print("PART 2 - Persisted dataset table relationships")


        # Part 3: Tables
        tableResults = results.get("dataDocumentationResult", {}).get("datasetResult", {}).get("tableResults", {})
        details["tableResults"] = tableResults
        if tableResults is not None:
            truncate_bigquery_table(f"{PROJECT_ID}.{metadata_dataset_id}.table_descriptions")
            truncate_bigquery_table(f"{PROJECT_ID}.{metadata_dataset_id}.table_column_descriptions")

        for i, tableResult in enumerate(tableResults):
          table_nm=""
          table_description=""

          table_fqn=tableResult.get("name", "")
          table_resource_uri_parts = table_fqn.split("/")
          table_nm=table_resource_uri_parts[8]
          table_description=tableResult.get("overview", "")

          row_data = {
                "name": table_nm,
                "description": table_description,
            }

          # Persist table description
          write_dict_to_bigquery(f"{PROJECT_ID}.{metadata_dataset_id}.table_descriptions", row_data)
          #print("PART 3 - Persisted table metadata")

          # Part 4: Persist table columns and descriptions
          tableColumnResults = tableResult.get("schema", {}).get("fields", [])
          details["tableColumnResults"] = tableColumnResults

          for i, tableColumnResult in enumerate(tableColumnResults):

            table_column_nm="None"
            table_column_description="None"

            table_column_nm=tableColumnResult.get("name")
            table_column_description=tableColumnResult.get("description")

            row_data = {
                  "table_name": table_nm,
                  "column_name": table_column_nm,
                  "column_description": table_column_description,
              }

            # Persist table column descriptions
            write_dict_to_bigquery(f"{PROJECT_ID}.{metadata_dataset_id}.table_column_descriptions", row_data)

          #print("PART 4 - Persisted table column metadata")

    except Exception as e:
        error_message = f"An error occurred during metadata generation: {e}"
        # Return a dictionary with an error status for ADK
        return {"status": "error", "error": error_message}

    return {"status": "success",
            "Response": f"Successfully saved the Data scans for the {dataset_type} dataset.",
            "Details": details
            }


### 2.5. Data engineering utils

In [ ]:
PIPELINE_ID = f"projects/{PROJECT_ID}/locations/{LOCATION}/repositories/{DATAFORM_REPO}/workspaces/{DATAFORM_WORKSPACE}"


# Instruction files
NAMING_CONVENTIONS_FILE = "01-naming-conventions.md"
NAMING_CONVENTIONS_CONTENT = """
Use the following naming convention when creating new objects:

* Tables:

    - Dimension: dim_[dimension_name]
    - Fact: fact_[fact_name]

* Views: vw_[view_name]

* Columns: snake_case (e.g., order_id, customer_name)

Always use ${ref()} to ensure dependency between nodes in dataform. ex: ${ref("referenced_sqlx_file")}

Make sure to create fine granular dimensions tables. Do not use left joins in defining dimension tables.
Make sure to create a dimension table for each table from the dataset that describes categories and types in the original dataset.
Tables that describe types or categories of different entities should be dimensions tables. Prefer more dimension over joining more tables to create one dimension table.

Consider that tables that record individual orders, stock movements, returns, and payments should be fact tables.
Consolidate more tables that store related information in the original dataset into fact tables such as orders, stock movements, returns, and payments.
There should be no unions used for the definitions of facts tables. Only joins.
In the dataform repository, place the definitions for the facts table in a `facts` folder and the definitions for dimensons in a `dims` folder.

For fact tables, the name should be the name of the table that is the main source of data from the operational database.
For example, if the name of the main source table is sales_orders, than the fact table name should be fact_sales_orders.

Add the string 'data warehouse' and source table names as tags in the config block of the .sqlx definition files.

Facts must not come from one source table. Add left joins in fact definitions.
Dimension tables must not be contained in the facts definitions.
"""



In [ ]:
def set_stage_for_star_schema_generation() -> dict:
    """
    1. Creates a Dataform repository, workspace and initializes the workspace.
    2. Generates a naming conventions markdown file and persists it to the dataform workspace
    3. Generates metadata for agentic grounding off of the Dataplex scans from module 2 and persists it to a markdown file in the dataform workspace
    """

    agent_grounding_content = ""
    naming_conventions_file = NAMING_CONVENTIONS_FILE
    naming_conventions_content = NAMING_CONVENTIONS_CONTENT

    # Generate auth token
    token = get_auth_token()

    # Delete the existing Dataform workspace
    delete_dataform_workspace(token, PROJECT_ID,LOCATION,DATAFORM_REPO,DATAFORM_WORKSPACE)

    # Create the Dataform workspace
    create_dataform_repository_and_workspace(
        token,
        PROJECT_ID,
        LOCATION,
        DATAFORM_REPO,
        DATAFORM_WORKSPACE
    )

    # Initialize the Dataform workspace
    initialize_dataform_workspace(
        token,
        PROJECT_ID,
        LOCATION,
        DATAFORM_REPO,
        DATAFORM_WORKSPACE
    )


    # Persist the latest Dataplex scan data to a separate dataset for use for agentic grounding
    print("Starting persisting scan results")
    # Commented due to bug https://buganizer.corp.google.com/issues/473862514
    persist_dataplex_scan_output_to_bq_tables("OLTP")
    print("Completed persisting scan results")

    # Generate agentic grounding instruction content
    description_df = read_bigquery_table(PROJECT_ID,OLTP_METADATA_DATASET_ID, "dataset_description")
    relationships_df = read_bigquery_table(PROJECT_ID,OLTP_METADATA_DATASET_ID, "dataset_table_relationships")
    tables_df = read_bigquery_table(PROJECT_ID,OLTP_METADATA_DATASET_ID, "table_descriptions")
    columns_df = read_bigquery_table(PROJECT_ID,OLTP_METADATA_DATASET_ID, "table_column_descriptions")

    agent_grounding_file = "02-agent-grounding.md"
    agent_grounding_content += "Description of the BigQuery dataset:"
    agent_grounding_content += "\n"
    agent_grounding_content = describe_dataset(description_df)
    agent_grounding_content += "\n"
    agent_grounding_content += "------------------------------------"
    agent_grounding_content += "\n"
    agent_grounding_content += "Description of the tables in the same dataset:"
    agent_grounding_content += "\n"
    agent_grounding_content += "\n".join(describe_tables(tables_df))
    agent_grounding_content += "\n"
    agent_grounding_content += "------------------------------------"
    agent_grounding_content += "\n"
    agent_grounding_content += "The relationships between tables in the same dataset:"
    agent_grounding_content += "\n"
    agent_grounding_content += "\n".join(describe_relationships(relationships_df))
    agent_grounding_content += "\n"
    agent_grounding_content += "------------------------------------"
    agent_grounding_content += "Description of the columns of the tables in the same dataset:"
    agent_grounding_content += describe_columns(columns_df)
    agent_grounding_content += "------------------------------------"


    try:

        # Delete the Dataform folder for instructions for the Data Engineering Agent
        result = delete_dataform_folder(
            token,
            PROJECT_ID,
            LOCATION,
            DATAFORM_REPO,
            DATAFORM_WORKSPACE,
            ".gdeagent/instructions"
        )

        # Create naming conventions instruction file
        result = create_dataform_instructions_file(
            token,
            PROJECT_ID,
            LOCATION,
            DATAFORM_REPO,
            DATAFORM_WORKSPACE,
            filename=naming_conventions_file,
            content=naming_conventions_content
        )

        # Create agenting grounding instruction file
        result = create_dataform_instructions_file(
              token,
              PROJECT_ID,
              LOCATION,
              DATAFORM_REPO,
              DATAFORM_WORKSPACE,
              filename=agent_grounding_file,
              content=agent_grounding_content
          )


    except Exception as e:
        error_message = f"An error occurred while doing the pre-work for the Data Engineering Agent: {e}"
        return {"status": "error", "error": error_message}

    return {"status": "success", "Response": f"Completed the pre-work for the Data Engineering Agent", "Details": result}


### 2.6. Create BigQuery tables to persist table metadata

In [ ]:
%%bigquery

CREATE TABLE `rscw_oltp_metadata_ds.dataset_description`
(
  dataset_description STRING
);

CREATE TABLE `rscw_oltp_metadata_ds.dataset_table_relationships`
(
  table_1 STRING,
  table_1_column STRING,
  table_2 STRING,
  table_2_column STRING,
  join_type STRING
);

CREATE TABLE `rscw_oltp_metadata_ds.table_column_descriptions`
(
  table_name STRING,
  column_name STRING,
  column_description STRING
);

CREATE TABLE `rscw_oltp_metadata_ds.table_descriptions`
(
  name STRING,
  description STRING
);

Executing query with job ID: c3468a2e-2d37-4c00-aca4-b32a21bf6bb4
Query executing: 0.21s


ERROR:
 400 GET https://bigquery.googleapis.com/bigquery/v2/projects/data-insights-quickstart/queries/c3468a2e-2d37-4c00-aca4-b32a21bf6bb4?maxResults=0&location=us-central1&prettyPrint=false: Already Exists: Table data-insights-quickstart:rscw_oltp_metadata_ds.dataset_description at [1:1]

Location: us-central1
Job ID: c3468a2e-2d37-4c00-aca4-b32a21bf6bb4



## 3. Complete the pre-work needed for the Data Engineering Agent to generate the star schema

In [ ]:
set_stage_for_star_schema_generation()

Attempting to delete workspace 'rscw-df-ws'...
Workspace 'rscw-df-ws' deleted successfully.
Repository 'rscw-df-repo' already exists.
Workspace 'rscw-df-ws' created successfully.
Failed to write workflow_settings.yaml: {
  "error": {
    "code": 500,
    "message": "An internal error has occurred (810dae4b-4297-4495-9fc7-e2ea352a5a2f)",
    "status": "INTERNAL"
  }
}

Starting persisting scan results
Starting saving the Data Insights scans for the OLTP dataset... ---
Fetching dataset documentation scan ---
Fetching scan results...
Base URL for API call for full results:https://dataplex.googleapis.com/v1/projects/data-insights-quickstart/locations/us-central1/dataScans/rscw-oltp-stg-ds-dataset-documentation-scan?view=FULL
Successfully fetched scan results.
Results:{'name': 'projects/data-insights-quickstart/locations/us-central1/dataScans/rscw-oltp-stg-ds-dataset-documentation-scan', 'uid': 'e1766208-708e-47e9-a6cb-edd5c9049238', 'displayName': 'rscw-oltp-stg-ds-dataset-documentation-sc

{'status': 'success',
 'Response': 'Completed the pre-work for the Data Engineering Agent',
 'Details': {}}

## This concludes the pre-work needed for Data Warehouse code generation. Proceed to the user manual for further instructions.